# M3 — LTV/CAC: Marketing Analytics para Startups LATAM

**Caso de uso:** EdTech de cursos profesionales online (B2C + B2B)  
*(Inspirado en Platzi — Colombia/LATAM, Crehana — Perú, Aprende Institute)*

---

## Objetivos del módulo

1. Calcular el LTV (Lifetime Value) por canal de adquisición
2. Calcular el CAC (Customer Acquisition Cost) por canal
3. Analizar el ratio LTV/CAC y el Payback Period
4. Visualizar qué canales son rentables y cuáles deben revisarse
5. Construir un análisis de cohortes de retención mes a mes
6. Generar recomendaciones de presupuesto de marketing

---

**Fórmulas clave:**
```
LTV  = ARPU × Gross Margin % / Churn Rate mensual
CAC  = Inversión en Marketing / Nuevos Clientes Adquiridos
LTV/CAC ≥ 3x → canal rentable (benchmark industria global)
Payback Period = CAC / (ARPU × Gross Margin %)
```

> **Repositorio:** [business-analytics-para-startups](https://github.com/scientificbroker/business-analytics-para-startups)

## 0. Instalación de dependencias

In [ ]:
# Instala las dependencias necesarias (ejecuta solo una vez en Google Colab)
%pip install pandas numpy matplotlib seaborn -q


## 1. Configuración de parámetros

Ajusta estos valores con los datos reales de tu negocio.

In [ ]:
# ════════════════════════════════════════════════
# PARÁMETROS CONFIGURABLES
# ════════════════════════════════════════════════

N_CLIENTES  = 1200
RANDOM_SEED = 42

# Gross Margin EdTech: 65-75% (infraestructura cloud, docentes, soporte)
GROSS_MARGIN = 0.68

# Canales y sus perfiles: (ARPU_media_USD, churn_mensual, costo_acq_USD)
PERFIL_CANAL = {
    'Organico/SEO':    (42, 0.08, 12),
    'Paid Social':     (35, 0.16, 45),
    'Referidos':       (48, 0.06, 18),
    'Email Marketing': (40, 0.10, 8),
    'Paid Search':     (38, 0.14, 52),
}

# Inversión mensual por canal y nuevos clientes adquiridos
INVERSION_MENSUAL   = [3500, 12000, 4200, 1800, 9500]   # USD/mes
CLIENTES_MES_PROM   = [290, 266, 240, 180, 100]         # nuevos clientes/mes

## 2. Importaciones

In [ ]:
%matplotlib inline
from IPython.display import display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')
PALETTE_CANALES = {
    'Organico/SEO':    '#2ECC71',
    'Paid Social':     '#E74C3C',
    'Referidos':       '#3498DB',
    'Email Marketing': '#F39C12',
    'Paid Search':     '#9B59B6',
}
plt.rcParams.update({'figure.figsize': (13, 7), 'font.size': 11,
                     'axes.titlesize': 13, 'axes.titleweight': 'bold'})
np.random.seed(RANDOM_SEED)

print('Modulo 3: LTV/CAC — EdTech LATAM')
print('Similar a: Platzi, Crehana, Aprende Institute, Descomplica')


## 3. Dataset de clientes y transacciones

El **perfil del canal** determina la calidad del cliente que trae.  
Los referidos traen clientes más baratos (bajo CAC) y con mejor retención (bajo churn) porque vienen con una recomendación personal de alguien que ya confía en el producto.

In [ ]:
CANALES    = list(PERFIL_CANAL.keys())
P_CANALES  = [0.30, 0.25, 0.20, 0.15, 0.10]

paises = ['México', 'Brasil', 'Argentina', 'Colombia', 'Chile', 'Perú']
p_pais = [0.22, 0.28, 0.15, 0.18, 0.10, 0.07]
planes = ['Mensual', 'Trimestral', 'Anual']
p_plan = [0.50, 0.30, 0.20]

fecha_inicio = datetime(2024, 10, 1)
fecha_fin    = datetime(2026, 4, 1)
delta_dias   = (fecha_fin - fecha_inicio).days

clientes = []
for i in range(N_CLIENTES):
    canal       = np.random.choice(CANALES, p=P_CANALES)
    arpu_b, churn_b, _ = PERFIL_CANAL[canal]
    fecha_adq   = fecha_inicio + timedelta(days=int(np.random.uniform(0, delta_dias)))
    plan        = np.random.choice(planes, p=p_plan)
    factor_plan = {'Mensual': 1.0, 'Trimestral': 0.90, 'Anual': 0.75}[plan]
    arpu        = arpu_b * factor_plan * np.random.uniform(0.85, 1.15)
    churn_m     = churn_b * np.random.uniform(0.8, 1.2)
    meses_max   = (fecha_fin - fecha_adq).days // 30
    meses_activo = max(1, min(int(np.random.geometric(churn_m)), meses_max))
    churned      = meses_activo < meses_max

    clientes.append({
        'cliente_id':    f'CLI-{i+1:05d}',
        'canal':         canal,
        'plan':          plan,
        'pais':          np.random.choice(paises, p=p_pais),
        'fecha_adq':     fecha_adq,
        'meses_activo':  meses_activo,
        'arpu_usd':      round(arpu, 2),
        'churn_rate_m':  round(churn_m, 4),
        'churned':       int(churned),
    })

df = pd.DataFrame(clientes)
df['revenue_total'] = df['arpu_usd'] * df['meses_activo']
df['mes_adq']       = df['fecha_adq'].dt.to_period('M').astype(str)

print(f'Dataset creado: {len(df)} clientes')
print(f'Revenue total generado: ${df["revenue_total"].sum():,.0f} USD')
print(f'Churn global: {df["churned"].mean():.1%}')

## 4. Tabla de CAC por canal

In [ ]:
df_costos = pd.DataFrame({
    'canal':               CANALES,
    'inversion_mensual':   INVERSION_MENSUAL,
    'n_clientes_mes_prom': CLIENTES_MES_PROM,
})
df_costos['cac_usd'] = (df_costos['inversion_mensual'] / df_costos['n_clientes_mes_prom']).round(2)

print('Costos de Adquisición por Canal (CAC):')
display(df_costos)

## 5. Cálculo LTV/CAC por canal

**Semáforo de rentabilidad:**
- **LTV/CAC ≥ 3x** → Canal rentable: escalar inversión
- **LTV/CAC 2x-3x** → Canal marginal: optimizar antes de escalar
- **LTV/CAC < 2x** → Canal no rentable: revisar estrategia o pausar

**Payback Period:**
- **≤ 6 meses** → excelente (el CAC se recupera muy rápido)
- **6-12 meses** → saludable para SaaS
- **> 12 meses** → riesgo de flujo de caja (necesitas más capital para crecer)

In [ ]:
metricas_canal = (
    df.groupby('canal')
    .agg(
        n_clientes   = ('cliente_id', 'count'),
        arpu_medio   = ('arpu_usd', 'mean'),
        churn_medio  = ('churn_rate_m', 'mean'),
        revenue_prom = ('revenue_total', 'mean'),
    )
    .reset_index()
)

# LTV = (ARPU × Gross Margin) / Churn mensual
metricas_canal['ltv_usd'] = (
    metricas_canal['arpu_medio'] * GROSS_MARGIN / metricas_canal['churn_medio']
).round(2)

metricas_canal = metricas_canal.merge(df_costos[['canal', 'cac_usd']], on='canal')
metricas_canal['ltv_cac_ratio']  = (metricas_canal['ltv_usd'] / metricas_canal['cac_usd']).round(2)
metricas_canal['payback_meses']  = (
    metricas_canal['cac_usd'] / (metricas_canal['arpu_medio'] * GROSS_MARGIN)
).round(1)

def semaforo(ratio):
    if ratio >= 3:   return 'RENTABLE'
    elif ratio >= 2: return 'MARGINAL'
    else:            return 'NO RENTABLE'

metricas_canal['estado'] = metricas_canal['ltv_cac_ratio'].apply(semaforo)

print('Tabla Maestra LTV/CAC por Canal:')
cols_show = ['canal', 'n_clientes', 'arpu_medio', 'churn_medio',
             'ltv_usd', 'cac_usd', 'ltv_cac_ratio', 'payback_meses', 'estado']
display(metricas_canal[cols_show].round(2))

## 6. Dashboard LTV/CAC — Visualización ejecutiva

Esta visualización está diseñada para **presentar a inversores** en el pitch deck.  
El scatter LTV vs CAC revela de un vistazo qué canales están sobre la línea de rentabilidad (3x).

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle('Dashboard LTV/CAC — EdTech LATAM\nAnálisis de Rentabilidad por Canal de Adquisición',
             fontsize=15, fontweight='bold', y=0.98)

gs = plt.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Plot 1: LTV vs CAC (scatter)
ax1 = fig.add_subplot(gs[0, 0])
colores = [PALETTE_CANALES[c] for c in metricas_canal['canal']]
ax1.scatter(metricas_canal['cac_usd'], metricas_canal['ltv_usd'],
            s=metricas_canal['n_clientes'] * 0.8, c=colores, alpha=0.8, edgecolors='white', lw=1)
for _, row in metricas_canal.iterrows():
    ax1.annotate(row['canal'].split('/')[0], (row['cac_usd'], row['ltv_usd']),
                 textcoords='offset points', xytext=(5, 5), fontsize=8)
max_cac = metricas_canal['cac_usd'].max() * 1.2
x_line  = np.linspace(0, max_cac, 100)
ax1.plot(x_line, x_line * 3, 'r--', lw=1.5, label='LTV/CAC = 3x (umbral)')
ax1.plot(x_line, x_line * 1, 'orange', linestyle=':', lw=1.5, label='LTV/CAC = 1x (break even)')
ax1.set_xlabel('CAC (USD)')
ax1.set_ylabel('LTV (USD)')
ax1.set_title('LTV vs CAC por Canal\n(tamaño = # clientes)')
ax1.legend(fontsize=8)

# Plot 2: Ratio LTV/CAC por canal
ax2 = fig.add_subplot(gs[0, 1])
sorted_df  = metricas_canal.sort_values('ltv_cac_ratio', ascending=True)
bar_colors = ['#2ECC71' if r >= 3 else '#F39C12' if r >= 2 else '#E74C3C'
              for r in sorted_df['ltv_cac_ratio']]
bars = ax2.barh(sorted_df['canal'], sorted_df['ltv_cac_ratio'],
                color=bar_colors, edgecolor='white', height=0.6)
ax2.axvline(3, color='green', linestyle='--', lw=2, label='Meta: 3x')
ax2.axvline(1, color='red', linestyle=':', lw=1.5, label='Break-even: 1x')
ax2.set_xlabel('Ratio LTV/CAC')
ax2.set_title('Ratio LTV/CAC por Canal\n(verde >= 3x = rentable)')
ax2.legend(fontsize=9)
for bar, val in zip(bars, sorted_df['ltv_cac_ratio']):
    ax2.text(val + 0.05, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}x', va='center', fontsize=9, fontweight='bold')

# Plot 3: Payback Period
ax3 = fig.add_subplot(gs[0, 2])
sorted_pb  = metricas_canal.sort_values('payback_meses')
pb_colors  = ['#2ECC71' if p <= 6 else '#F39C12' if p <= 12 else '#E74C3C'
              for p in sorted_pb['payback_meses']]
bars3 = ax3.bar(sorted_pb['canal'], sorted_pb['payback_meses'],
                color=pb_colors, edgecolor='white')
ax3.axhline(6,  color='green',  linestyle='--', lw=1.5, label='Meta: 6 meses')
ax3.axhline(12, color='orange', linestyle=':', lw=1.5, label='Maximo saludable: 12 meses')
ax3.set_ylabel('Meses para recuperar CAC')
ax3.set_title('Payback Period por Canal\n(verde <= 6 meses = excelente)')
ax3.tick_params(axis='x', rotation=30)
ax3.legend(fontsize=8)
for bar, val in zip(bars3, sorted_pb['payback_meses']):
    ax3.text(bar.get_x() + bar.get_width()/2, val + 0.1,
             f'{val:.0f}m', ha='center', fontsize=9)

# Plot 4: Revenue acumulado proyectado
ax4 = fig.add_subplot(gs[1, :2])
meses = np.arange(1, 25)
for _, row in metricas_canal.iterrows():
    rev_acum = np.cumsum([row['n_clientes'] * row['arpu_medio'] * (1 - row['churn_medio'])**m
                          for m in meses])
    ax4.plot(meses, rev_acum / 1000, label=row['canal'],
             color=PALETTE_CANALES[row['canal']], lw=2.5, marker='o', markersize=3)
ax4.set_xlabel('Mes')
ax4.set_ylabel('Revenue Acumulado (USD miles)')
ax4.set_title('Proyección de Revenue Acumulado por Canal (24 meses)')
ax4.legend(fontsize=9)
ax4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}K'))

# Plot 5: Resumen ejecutivo en texto
ax5 = fig.add_subplot(gs[1, 2])
ax5.axis('off')
mejor_canal = metricas_canal.loc[metricas_canal['ltv_cac_ratio'].idxmax(), 'canal']
peor_canal  = metricas_canal.loc[metricas_canal['ltv_cac_ratio'].idxmin(), 'canal']
mejor_ratio = metricas_canal['ltv_cac_ratio'].max()
peor_ratio  = metricas_canal['ltv_cac_ratio'].min()

resumen = (
    f'RESUMEN EJECUTIVO\n\n'
    f'Gross Margin asumido: {GROSS_MARGIN:.0%}\n'
    f'Benchmark LATAM EdTech:\n'
    f'  LTV/CAC meta: >= 3.0x\n'
    f'  Payback meta: <= 6 meses\n\n'
    f'MEJOR canal: {mejor_canal}\n'
    f'  LTV/CAC = {mejor_ratio:.1f}x\n\n'
    f'PEOR canal: {peor_canal}\n'
    f'  LTV/CAC = {peor_ratio:.1f}x\n\n'
    f'Accion: escalar presupuesto\n'
    f'en {mejor_canal} y revisar\n'
    f'estrategia de {peor_canal}.'
)
ax5.text(0.05, 0.95, resumen, transform=ax5.transAxes,
         fontsize=10, verticalalignment='top',
         bbox=dict(boxstyle='round,pad=0.8', facecolor='#EBF5FB', alpha=0.9))

plt.savefig('M3_ltv_cac_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Análisis de cohortes de retención

Las **cohortes** muestran qué porcentaje de clientes adquiridos en un mes determinado siguen activos en los meses siguientes.

**Métricas que piden los VCs:**
- **M+3:** % de clientes de una cohorte activos al mes 3
- **M+6:** % activos al mes 6
- **Benchmark EdTech LATAM:** M+3 ~60-70%, M+6 ~45-60%

Un "kink" (caída brusca) en un mes específico indica un punto de quiebre en el onboarding o en el valor percibido.

In [ ]:
df['cohort'] = df['fecha_adq'].dt.to_period('M').astype(str)
COHORTES     = sorted(df['cohort'].unique())[-8:]
MAX_MES      = 8

cohort_data = {}
for cohorte in COHORTES:
    clientes_cohorte = df[df['cohort'] == cohorte]
    n_total = len(clientes_cohorte)
    fila = [100.0]
    for mes in range(1, MAX_MES + 1):
        activos = (clientes_cohorte['meses_activo'] >= mes).sum()
        fila.append(round(activos / n_total * 100, 1))
    cohort_data[cohorte] = fila

cohort_df = pd.DataFrame(cohort_data, index=[f'M+{i}' for i in range(MAX_MES + 1)]).T

fig, ax = plt.subplots(figsize=(12, 6))
mask = cohort_df.isnull()
sns.heatmap(cohort_df, annot=True, fmt='.0f', cmap='YlGn',
            mask=mask, ax=ax, linewidths=0.5,
            vmin=0, vmax=100,
            cbar_kws={'label': '% Clientes Activos'})
ax.set_title('Análisis de Cohortes — Retención Mensual (%)\nEdTech LATAM', fontsize=13, fontweight='bold')
ax.set_xlabel('Mes desde Adquisición')
ax.set_ylabel('Cohorte de Adquisición')
plt.tight_layout()
plt.savefig('M3_cohortes_retencion.png', dpi=150, bbox_inches='tight')
plt.show()

m3_prom = cohort_df['M+3'].mean()
m6_prom = cohort_df['M+6'].mean() if 'M+6' in cohort_df.columns else None
print(f'Retención promedio M+3: {m3_prom:.1f}%  (Benchmark LATAM EdTech: ~60-70%)')
if m6_prom:
    print(f'Retención promedio M+6: {m6_prom:.1f}%  (Benchmark LATAM EdTech: ~45-60%)')

## 8. Conclusiones ejecutivas y próximos pasos

In [ ]:
rentables = metricas_canal[metricas_canal['ltv_cac_ratio'] >= 3]['canal'].tolist()
no_rent   = metricas_canal[metricas_canal['ltv_cac_ratio'] < 2]['canal'].tolist()

print('=' * 65)
print('CONCLUSIONES EJECUTIVAS — LTV/CAC EDTECH LATAM')
print('=' * 65)
print(f"""
HALLAZGOS CLAVE:

  1. Canales rentables (LTV/CAC >= 3x): {', '.join(rentables) if rentables else 'Ninguno aún'}
     Escalar inversión en estos canales tiene ROI demostrado.

  2. Canales a revisar (LTV/CAC < 2x): {', '.join(no_rent) if no_rent else 'Ninguno en esta zona'}
     Evaluar si el problema es el CAC alto o la retención baja.
     Solución: mejorar targeting (baja CAC) o mejorar onboarding (mejora LTV).

  3. El canal Referidos tiene el churn más bajo: los clientes que llegan
     recomendados tienen más fit con el producto.
     → Diseñar un programa de referidos activo con incentivos.

  4. Retención M+3 del {m3_prom:.0f}%: si está por debajo del 60%, hay un problema
     de onboarding o de valor percibido en las primeras semanas.

ACCIONES INMEDIATAS:
  → Calcular estos KPIs con tus datos reales cada mes.
  → Si LTV/CAC < 3x en todos los canales, no escales marketing: arregla retención primero.
  → Define un MRR objetivo y trabaja hacia atrás: ¿cuántos clientes con qué CAC?

PROXIMO MODULO RECOMENDADO:
  Para segmentar clientes por valor real → M4 (KMeans + RFM)
  Para proyectar revenue de los próximos 12 meses → M5 (Prophet Forecasting)
""")